In [ ]:
!pip install -q -U transformers accelerate bitsandbytes bert-score pandas

In [ ]:
import os
import time
import gc
import logging
import warnings

import torch
import pandas as pd

from collections import Counter

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from bert_score import score as bert_score


# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "/kaggle/input/YOUR-DATASET/CHQ_SUM_dataset_2000.csv"

MODEL_ID = "md-nishat-008/TigerLLM-9B-it"

OUTPUT_DIR = "/kaggle/working/bangla_chq_tigerllm_8bit"

# None = full dataset
# Set 10, 50, 100 etc. for testing
NUM_SAMPLES = None

# Start with 1 on Kaggle.
# Increase to 2 if GPU memory allows.
BATCH_SIZE = 1

MAX_INPUT_LENGTH = 2048
MAX_NEW_TOKENS = 256


# ============================================================
# SETTINGS
# ============================================================

os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

warnings.filterwarnings("ignore")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)


# ============================================================
# PAPER BASELINE SCORES
# ============================================================

PAPER_SCORES = {
    "ROUGE-1": 50.05,
    "ROUGE-2": 29.11,
    "ROUGE-L": 48.35,
    "BERTScore": 89.91,
}


# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are a Bengali medical question summarization model.

Given a Bengali health-related question asked by a patient,
generate a concise summary.

Rules:
- Retain all medically relevant information required to answer
  the question accurately.
- Be as concise as possible without discarding essential information.
- Preserve symptom details, duration, medications mentioned,
  and the core question.
- Output ONLY the Bengali summary.
- Do not include explanations, labels, quotation marks,
  or any extra text.
""".strip()


def build_user_message(question):
    return f"Summarize this Bengali health question:\n{question}"


# ============================================================
# ROUGE FUNCTIONS
# ============================================================

def _ngrams(tokens, n):

    return Counter(
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    )


def _rouge_n(pred_tokens, ref_tokens, n):

    pred_ng = _ngrams(pred_tokens, n)
    ref_ng = _ngrams(ref_tokens, n)

    overlap = sum(
        (pred_ng & ref_ng).values()
    )

    precision = (
        overlap
        / max(sum(pred_ng.values()), 1)
    )

    recall = (
        overlap
        / max(sum(ref_ng.values()), 1)
    )

    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-9)
    )

    return f1


def _lcs_len(a, b):

    m = len(a)
    n = len(b)

    previous = [0] * (n + 1)

    for i in range(1, m + 1):

        current = [0] * (n + 1)

        for j in range(1, n + 1):

            if a[i - 1] == b[j - 1]:

                current[j] = (
                    previous[j - 1] + 1
                )

            else:

                current[j] = max(
                    previous[j],
                    current[j - 1]
                )

        previous = current

    return previous[n]


def _rouge_l(pred_tokens, ref_tokens):

    lcs = _lcs_len(
        pred_tokens,
        ref_tokens
    )

    precision = (
        lcs / max(len(pred_tokens), 1)
    )

    recall = (
        lcs / max(len(ref_tokens), 1)
    )

    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-9)
    )

    return f1


# ============================================================
# LOAD TIGERLLM IN 8-BIT
# ============================================================

def load_model():

    if not torch.cuda.is_available():

        raise RuntimeError(
            "GPU is not enabled. "
            "In Kaggle go to Settings -> Accelerator -> GPU."
        )


    print("GPU:", torch.cuda.get_device_name(0))

    print("\nLoading tokenizer...")


    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True
    )


    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )


    tokenizer.padding_side = "left"


    print(
        "Loading TigerLLM with "
        "8-bit quantization..."
    )


    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True
    )


    model = AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        quantization_config=quantization_config,

        device_map="auto",

        torch_dtype=torch.float16,

        trust_remote_code=True
    )


    model.eval()


    print(
        "\nTigerLLM loaded successfully."
    )


    try:

        memory = (
            model.get_memory_footprint()
            / 1024**3
        )

        print(
            f"Model memory footprint: "
            f"{memory:.2f} GB"
        )

    except Exception:
        pass


    return model, tokenizer


# ============================================================
# BUILD CHAT PROMPT
# ============================================================

def build_prompt(
    tokenizer,
    question
):

    messages = [

        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },

        {
            "role": "user",
            "content":
                build_user_message(
                    question
                )
        }

    ]


    try:

        prompt = (
            tokenizer.apply_chat_template(

                messages,

                tokenize=False,

                add_generation_prompt=True

            )
        )


    except Exception:

        prompt = f"""
{SYSTEM_PROMPT}

User:
{build_user_message(question)}

Assistant:
""".strip()


    return prompt


# ============================================================
# CLEAN MODEL OUTPUT
# ============================================================

def clean_prediction(text):

    if text is None:
        return ""


    text = str(text).strip()


    prefixes = [

        "Summary:",
        "Answer:",
        "Output:",
        "সারাংশ:",
        "উত্তর:"
    ]


    for prefix in prefixes:

        if text.lower().startswith(
            prefix.lower()
        ):

            text = text[
                len(prefix):
            ].strip()


    # Sometimes model generates extra lines
    lines = [

        line.strip()

        for line in text.splitlines()

        if line.strip()
    ]


    if lines:

        text = lines[0]


    # Remove quotation marks
    text = text.strip(
        "\"'“”‘’"
    )


    return text.strip()


# ============================================================
# RUN ONE BATCH
# ============================================================

@torch.inference_mode()
def run_batch(
    model,
    tokenizer,
    batch_rows
):

    prompts = [

        build_prompt(
            tokenizer,
            row["question"]
        )

        for row in batch_rows
    ]


    inputs = tokenizer(

        prompts,

        return_tensors="pt",

        padding=True,

        truncation=True,

        max_length=MAX_INPUT_LENGTH

    )


    # First GPU used by model
    device = torch.device(
        "cuda:0"
    )


    inputs = {

        key: value.to(device)

        for key, value
        in inputs.items()

    }


    torch.cuda.synchronize()


    start_time = time.monotonic()


    outputs = model.generate(

        **inputs,

        max_new_tokens=MAX_NEW_TOKENS,

        do_sample=False,

        pad_token_id=
            tokenizer.pad_token_id,

        eos_token_id=
            tokenizer.eos_token_id,

        use_cache=True

    )


    torch.cuda.synchronize()


    elapsed = (
        time.monotonic()
        - start_time
    )


    per_item_latency = (
        elapsed
        / len(batch_rows)
    )


    # Full padded prompt length
    prompt_length = (
        inputs["input_ids"].shape[1]
    )


    generated_tokens = (
        outputs[
            :,
            prompt_length:
        ]
    )


    decoded = tokenizer.batch_decode(

        generated_tokens,

        skip_special_tokens=True

    )


    results = []


    for row, generated_text in zip(
        batch_rows,
        decoded
    ):

        prediction = clean_prediction(
            generated_text
        )


        results.append({

            "dataset_index":
                row["dataset_index"],

            "id":
                row["id"],

            "question":
                row["question"],

            "reference":
                row["reference"],

            "prediction":
                prediction,

            "raw_output":
                generated_text,

            "latency":
                per_item_latency,

            "error":
                None if prediction
                else "empty_output"

        })


    return results


# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def checkpoint_path():

    return os.path.join(
        OUTPUT_DIR,
        "checkpoint_tigerllm_8bit.csv"
    )


def load_checkpoint():

    path = checkpoint_path()


    if os.path.exists(path):

        try:

            return pd.read_csv(
                path
            )

        except Exception:

            return None


    return None


def save_checkpoint(rows):

    pd.DataFrame(
        rows
    ).to_csv(

        checkpoint_path(),

        index=False,

        encoding="utf-8-sig"
    )


# ============================================================
# RUN MODEL ON DATASET
# ============================================================

def run_model_on_dataset(
    model,
    tokenizer,
    df
):

    existing = load_checkpoint()


    rows_done = (

        existing.to_dict("records")

        if existing is not None

        else []

    )


    completed_indices = set()


    for row in rows_done:

        try:

            completed_indices.add(

                int(
                    row[
                        "dataset_index"
                    ]
                )

            )

        except Exception:
            pass


    remaining_rows = []


    for index, row in df.iterrows():

        if (
            index
            not in completed_indices
        ):

            remaining_rows.append({

                "dataset_index":
                    index,

                "id":
                    row.get(
                        "id",
                        index
                    ),

                "question":
                    str(
                        row["question"]
                    ).strip(),

                "reference":
                    str(
                        row["summary"]
                    ).strip(),

            })


    if not remaining_rows:

        print(
            "\nAlready complete — "
            "loaded from checkpoint."
        )

        return rows_done


    total_batches = (

        len(remaining_rows)
        + BATCH_SIZE
        - 1

    ) // BATCH_SIZE


    for batch_number in range(
        total_batches
    ):

        start = (
            batch_number
            * BATCH_SIZE
        )

        end = min(

            start + BATCH_SIZE,

            len(remaining_rows)

        )


        batch_rows = (
            remaining_rows[
                start:end
            ]
        )


        print(

            f"\nBatch "
            f"{batch_number + 1}/"
            f"{total_batches}"

            f" | Rows "

            f"{batch_rows[0]['dataset_index']}"
            f"-"
            f"{batch_rows[-1]['dataset_index']}"

        )


        try:

            batch_results = run_batch(

                model,

                tokenizer,

                batch_rows

            )


        except torch.cuda.OutOfMemoryError:

            print(
                "CUDA OOM. "
                "Retrying one row at a time..."
            )


            torch.cuda.empty_cache()


            batch_results = []


            for row in batch_rows:

                try:

                    result = run_batch(

                        model,

                        tokenizer,

                        [row]

                    )

                    batch_results.extend(
                        result
                    )


                except Exception as e:

                    batch_results.append({

                        **row,

                        "prediction":
                            "",

                        "raw_output":
                            "",

                        "latency":
                            None,

                        "error":
                            str(e)

                    })


        except Exception as e:

            print(
                f"Batch error: {e}"
            )


            batch_results = [

                {

                    **row,

                    "prediction":
                        "",

                    "raw_output":
                        "",

                    "latency":
                        None,

                    "error":
                        str(e)

                }

                for row
                in batch_rows

            ]


        for result in batch_results:

            print(
                f"[{result['dataset_index']}] "
                f"{result['prediction'][:100]}"
            )


        rows_done.extend(
            batch_results
        )


        rows_done = sorted(

            rows_done,

            key=lambda x:
                int(
                    x[
                        "dataset_index"
                    ]
                )

        )


        save_checkpoint(
            rows_done
        )


        torch.cuda.empty_cache()


    return rows_done


# ============================================================
# COMPUTE METRICS
# ============================================================

def compute_metrics(rows):

    predictions = []
    references = []
    latencies = []


    for row in rows:

        prediction = str(
            row.get(
                "prediction",
                ""
            )
        ).strip()


        reference = str(
            row.get(
                "reference",
                ""
            )
        ).strip()


        error = row.get(
            "error"
        )


        valid_error = (

            error is None

            or str(error).strip() == ""

            or str(error).lower()
            in {
                "nan",
                "none"
            }

        )


        if (
            prediction
            and reference
            and valid_error
        ):

            predictions.append(
                prediction
            )

            references.append(
                reference
            )


        latency = row.get(
            "latency"
        )


        if (
            latency is not None
            and str(latency).lower()
            != "nan"
        ):

            try:

                latencies.append(
                    float(latency)
                )

            except Exception:
                pass


    if not predictions:

        print(
            "No valid predictions found."
        )

        return {}


    print(
        f"\nEvaluating "
        f"{len(predictions)} "
        f"valid predictions..."
    )


    # ========================================================
    # ROUGE
    # ========================================================

    rouge1_scores = []
    rouge2_scores = []
    rougel_scores = []


    for pred, ref in zip(
        predictions,
        references
    ):

        pred_tokens = pred.split()
        ref_tokens = ref.split()


        rouge1_scores.append(

            _rouge_n(
                pred_tokens,
                ref_tokens,
                1
            )

        )


        rouge2_scores.append(

            _rouge_n(
                pred_tokens,
                ref_tokens,
                2
            )

        )


        rougel_scores.append(

            _rouge_l(
                pred_tokens,
                ref_tokens
            )

        )


    rouge1 = (
        sum(rouge1_scores)
        / len(rouge1_scores)
    ) * 100


    rouge2 = (
        sum(rouge2_scores)
        / len(rouge2_scores)
    ) * 100


    rougel = (
        sum(rougel_scores)
        / len(rougel_scores)
    ) * 100


    # ========================================================
    # BERTSCORE
    # ========================================================

    predictions_trunc = [

        " ".join(
            p.split()[:200]
        )

        for p in predictions
    ]


    references_trunc = [

        " ".join(
            r.split()[:200]
        )

        for r in references
    ]


    print(
        "\nComputing BERTScore..."
    )


    bert_device = (

        "cuda"

        if torch.cuda.is_available()

        else "cpu"

    )


    P, R, F1 = bert_score(

        predictions_trunc,

        references_trunc,

        model_type=
            "xlm-roberta-base",

        lang="bn",

        device=bert_device,

        verbose=True

    )


    bertscore = (
        F1.mean().item()
        * 100
    )


    return {

        "ROUGE-1":
            rouge1,

        "ROUGE-2":
            rouge2,

        "ROUGE-L":
            rougel,

        "BERTScore":
            bertscore,

        "total":
            len(predictions),

        "avg_latency":
            (
                sum(latencies)
                / len(latencies)

                if latencies

                else None
            )

    }


# ============================================================
# MAIN
# ============================================================

def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # ========================================================
    # LOAD DATASET
    # ========================================================

    df = pd.read_csv(
        CSV_PATH
    )


    required = {
        "question",
        "summary"
    }


    if not required.issubset(
        df.columns
    ):

        raise ValueError(

            "CSV must contain "
            "'question' and "
            "'summary' columns"

        )


    df = df.dropna(

        subset=[
            "question",
            "summary"
        ]

    ).reset_index(
        drop=True
    )


    df["question"] = (

        df["question"]
        .astype(str)
        .str.strip()

    )


    df["summary"] = (

        df["summary"]
        .astype(str)
        .str.strip()

    )


    df = df[

        (df["question"] != "")

        &

        (df["summary"] != "")

    ].reset_index(
        drop=True
    )


    if NUM_SAMPLES is not None:

        df = df.head(
            NUM_SAMPLES
        ).reset_index(
            drop=True
        )


    print(
        f"\nDataset: "
        f"{len(df)} "
        f"question-summary pairs"
    )


    # ========================================================
    # LOAD TIGERLLM
    # ========================================================

    model, tokenizer = load_model()


    # ========================================================
    # RUN GENERATION
    # ========================================================

    rows = run_model_on_dataset(

        model,

        tokenizer,

        df

    )


    # ========================================================
    # SAVE PREDICTIONS
    # ========================================================

    predictions_path = os.path.join(

        OUTPUT_DIR,

        "predictions.csv"

    )


    pd.DataFrame(
        rows
    ).to_csv(

        predictions_path,

        index=False,

        encoding="utf-8-sig"

    )


    print(
        "\nGeneration complete."
    )


    # ========================================================
    # IMPORTANT:
    # REMOVE TIGERLLM BEFORE BERTSCORE
    # ========================================================

    print(
        "\nUnloading TigerLLM "
        "before BERTScore..."
    )


    del model
    del tokenizer


    gc.collect()


    torch.cuda.empty_cache()


    # ========================================================
    # METRICS
    # ========================================================

    metrics = compute_metrics(
        rows
    )


    if not metrics:

        return


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    results_df = pd.DataFrame([{

        "model":
            MODEL_ID,

        "quantization":
            "8-bit",

        "ROUGE-1":
            round(
                metrics["ROUGE-1"],
                2
            ),

        "ROUGE-2":
            round(
                metrics["ROUGE-2"],
                2
            ),

        "ROUGE-L":
            round(
                metrics["ROUGE-L"],
                2
            ),

        "BERTScore":
            round(
                metrics["BERTScore"],
                2
            ),

        "total_sentences":
            metrics["total"],

        "avg_latency_s":
            (
                round(
                    metrics[
                        "avg_latency"
                    ],
                    3
                )

                if metrics[
                    "avg_latency"
                ] is not None

                else None
            )

    }])


    results_path = os.path.join(

        OUTPUT_DIR,

        "benchmark_results.csv"

    )


    results_df.to_csv(

        results_path,

        index=False,

        encoding="utf-8-sig"

    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        "\n── Evaluation Results ──"
    )


    print(

        f"{'Metric':<12} "
        f"{'TigerLLM':>10} "
        f"{'Paper':>10} "
        f"{'Diff':>10}"

    )


    print(
        "-" * 45
    )


    for metric in [

        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "BERTScore"

    ]:

        yours = metrics[
            metric
        ]

        paper = PAPER_SCORES[
            metric
        ]

        diff = yours - paper


        print(

            f"{metric:<12} "
            f"{yours:>10.2f} "
            f"{paper:>10.2f} "
            f"{diff:>+10.2f}"

        )


    print(
        f"\nTotal evaluated: "
        f"{metrics['total']}"
    )


    if (
        metrics["avg_latency"]
        is not None
    ):

        print(

            f"Average latency: "
            f"{metrics['avg_latency']:.2f}s"

        )


    print(
        f"\nPredictions: "
        f"{predictions_path}"
    )


    print(
        f"Results: "
        f"{results_path}"
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()